# Fantasy Football Data Pipeline

Pulls and joins raw NFL data from `nflreadpy` to produce `raw_dataset.csv`, the input for `features.ipynb`.

**Run order:** execute all cells top-to-bottom, or use the **Master Rebuild** cell (Part 3/4 boundary) to reconstruct `df` cleanly before running Part 4 steps.

**Output:** `fantasy/raw_dataset.csv` (~34,900 rows x 84 columns)

## Project Roadmap

Phase 1 ✅ Done
└── Local pipeline → per-position XGBoost models → Streamlit fantasy tab
    Note: betting model (separate system) uses Ensemble fixed75 as primary + XGBoost, Ridge, LightGBM as direction voters

Phase 2
└── Cloud storage (S3/GCS) for data + model artifacts

Phase 3
└── Databricks pipeline with Delta Lake + Spark

Phase 4
└── K-Means clustering → player archetypes + breakout signals
    └── Learn: unsupervised learning, feature scaling, cluster evaluation

Phase 5
└── Neural network as alternative points model
    └── Note: feedforward MLP (256→128→64, Huber loss) already deployed for the *betting* model;
         LSTM or similar for the *fantasy* model remains future work

Phase 6 ✅ Done (betting model)
└── LLM explanation layer — ReActAgent via LlamaIndex + Anthropic Claude API
    Live for betting predictions; fantasy explanation layer remains future work

Phase 7
└── MLflow + Evidently monitoring
    └── Learn: ML lifecycle, drift detection

Phase 8 (optional)
└── Palantir Foundry ontology

## Part 1 — Raw Data Loading

### Step 1 — Player Stats

Loads per-player weekly stats from `nfl.load_player_stats()` for QB/RB/WR/TE. Filters to regular season games only.

In [ ]:
import nflreadpy as nfl
import pandas as pd

# nflreadpy returns Polars DataFrames — we'll convert to pandas right after loading
# Seasons to pull
SEASONS = list(range(2018, 2026))

# Scoring format
SCORING = "half_ppr"  # change to "std" or "ppr" if needed

SCORING_WEIGHTS = {
    "half_ppr": {
        "pass_yd": 0.04, "pass_td": 4,  "pass_int": -2,
        "rush_yd": 0.1,  "rush_td": 6,
        "rec":     0.5,  "rec_yd": 0.1, "rec_td": 6,
        "fumble":  -2
    }
}

print("✅ Config ready")

In [74]:
player_stats = nfl.load_player_stats(SEASONS).to_pandas()

# Regular season only, skill positions only
player_stats = player_stats[player_stats["season_type"] == "REG"]
player_stats = player_stats[player_stats["position"].isin(["QB", "RB", "WR", "TE"])]

print(player_stats.shape)
print(player_stats.columns.tolist())

(34906, 115)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tack

### Step 2 — Fantasy Football Opportunity (FFO)

Loads expected points and over/underperformance signals from `nfl.load_ff_opportunity()`. Coverage is partial for low-usage players — nulls are filled with 0 later.

In [75]:
ff_opp = nfl.load_ff_opportunity(SEASONS).to_pandas()

print(ff_opp.shape)
print(ff_opp.columns.tolist())

(36063, 159)
['season', 'posteam', 'week', 'game_id', 'player_id', 'full_name', 'position', 'pass_attempt', 'rec_attempt', 'rush_attempt', 'pass_air_yards', 'rec_air_yards', 'pass_completions', 'receptions', 'pass_completions_exp', 'receptions_exp', 'pass_yards_gained', 'rec_yards_gained', 'rush_yards_gained', 'pass_yards_gained_exp', 'rec_yards_gained_exp', 'rush_yards_gained_exp', 'pass_touchdown', 'rec_touchdown', 'rush_touchdown', 'pass_touchdown_exp', 'rec_touchdown_exp', 'rush_touchdown_exp', 'pass_two_point_conv', 'rec_two_point_conv', 'rush_two_point_conv', 'pass_two_point_conv_exp', 'rec_two_point_conv_exp', 'rush_two_point_conv_exp', 'pass_first_down', 'rec_first_down', 'rush_first_down', 'pass_first_down_exp', 'rec_first_down_exp', 'rush_first_down_exp', 'pass_interception', 'rec_interception', 'pass_interception_exp', 'rec_interception_exp', 'rec_fumble_lost', 'rush_fumble_lost', 'pass_fantasy_points_exp', 'rec_fantasy_points_exp', 'rush_fantasy_points_exp', 'pass_fantasy_p

### Step 3 — Schedules & Game Context

Loads Vegas lines, weather, home/away, and rest days from `nfl.load_schedules()`. Filters to regular season games.

In [76]:
schedules = nfl.load_schedules(SEASONS).to_pandas()

# Regular season only
schedules = schedules[schedules["game_type"] == "REG"]

print(schedules.shape)
print(schedules.columns.tolist())

(1615, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [77]:
ps_cols = [
    # identifiers
    "player_id", "player_display_name", "position", "team",
    "opponent_team", "season", "week", "game_id",
    # target variable (already calculated for us)
    "fantasy_points", "fantasy_points_ppr",
    # passing
    "completions", "attempts", "passing_yards", "passing_tds",
    "passing_interceptions", "passing_air_yards", "passing_epa",
    # rushing
    "carries", "rushing_yards", "rushing_tds",
    "rushing_fumbles_lost", "rushing_epa",
    # receiving
    "receptions", "targets", "receiving_yards", "receiving_tds",
    "receiving_fumbles_lost", "receiving_air_yards",
    "receiving_yards_after_catch", "receiving_epa",
    # usage/opportunity
    "target_share", "air_yards_share", "wopr", "racr",
]

player_stats_clean = player_stats[ps_cols].copy()

print(player_stats_clean.shape)
player_stats_clean.tail()

(34906, 34)


,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,...,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_epa,target_share,air_yards_share,wopr,racr
111553,00-0040730,RJ Harvey,RB,DEN,LAC,2025,18,2025_18_LAC_DEN,3.30,4.30,...,5,0,0,-5,8,-3.856058,0.173913,-0.192308,0.126254,-1.000000
111555,00-0040734,TreVeyon Henderson,RB,NE,MIA,2025,18,2025_18_MIA_NE,17.30,17.30,...,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN
111556,00-0040735,Luther Burden III,WR,CHI,DET,2025,18,2025_18_DET_CHI,4.50,7.50,...,35,0,0,12,27,2.777708,0.137931,0.046154,0.239204,2.916667
111558,00-0040743,Tyler Shough,QB,NO,ATL,2025,18,2025_18_NO_ATL,21.76,21.76,...,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN
111563,00-0040782,Isaiah Bond,WR,CLE,CIN,2025,18,2025_18_CLE_CIN,1.70,2.70,...,13,0,0,48,0,0.556947,0.095238,0.244898,0.314286,0.270833


In [78]:
# ff_opp gives us expected points and the diff (actual - expected)
# total_fantasy_points_diff = actual minus expected = our over/underperformance signal
ffo_cols = [
    "player_id", "season", "week", "game_id",
    "total_fantasy_points",        # actual (their version)
    "total_fantasy_points_exp",    # expected based on opportunities
    "total_fantasy_points_diff",   # actual minus expected (key feature)
    "rec_attempt",                 # targets from their model
    "rush_attempt",
    "rec_yards_gained_exp",
    "rush_yards_gained_exp",
    "rec_touchdown_exp",
    "rush_touchdown_exp",
]

ffo_clean = ff_opp[ffo_cols].copy()
ffo_clean = ffo_clean.rename(columns={
    "total_fantasy_points":      "ffo_actual_pts",
    "total_fantasy_points_exp":  "ffo_expected_pts",
    "total_fantasy_points_diff": "ffo_pts_diff",   # positive = outperforming, negative = underperforming
})

print(ffo_clean.shape)
ffo_clean.head()

(36063, 13)


,player_id,season,week,game_id,ffo_actual_pts,ffo_expected_pts,ffo_pts_diff,rec_attempt,rush_attempt,rec_yards_gained_exp,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp
0,00-0031345,2020,1.0,2020_01_ARI_SF,19.26,17.66,1.60,0.0,1.0,0.00,7.04,0.00,0.00
1,00-0033288,2020,1.0,2020_01_ARI_SF,9.30,10.28,-0.98,5.0,1.0,38.61,5.87,0.23,0.09
2,00-0035228,2020,1.0,2020_01_ARI_SF,26.30,19.53,6.77,0.0,13.0,0.00,75.43,0.00,0.07
3,00-0030564,2020,1.0,2020_01_ARI_SF,29.10,24.03,5.07,16.0,0.0,118.96,0.00,0.10,0.00
4,00-0022921,2020,1.0,2020_01_ARI_SF,7.40,7.22,0.18,5.0,0.0,32.60,0.00,0.01,0.00


## Part 2 — Base DataFrame Assembly

In [79]:
home = schedules[["season", "week", "home_team", "away_team",
                   "total_line", "spread_line",
                   "wind", "temp", "roof", "surface",
                   "home_rest", "away_rest"]].copy()

home = home.rename(columns={"home_team": "team", "away_team": "opponent"})
home["implied_team_total"] = (home["total_line"] - home["spread_line"]) / 2
home["is_home"] = 1
home["days_rest"] = home["home_rest"]
home["is_dome"] = home["roof"].isin(["dome", "closed", "retractable"]).astype(int)
home["effective_wind"] = home["wind"].fillna(0) * (1 - home["is_dome"])
home["effective_temp"] = home["temp"].where(home["is_dome"] == 0, other=70).fillna(65)

away = schedules[["season", "week", "away_team", "home_team",
                   "total_line", "spread_line",
                   "wind", "temp", "roof", "surface",
                   "home_rest", "away_rest"]].copy()

away = away.rename(columns={"away_team": "team", "home_team": "opponent"})
away["implied_team_total"] = (away["total_line"] + away["spread_line"]) / 2
away["is_home"] = 0
away["days_rest"] = away["away_rest"]
away["is_dome"] = away["roof"].isin(["dome", "closed", "retractable"]).astype(int)
away["effective_wind"] = away["wind"].fillna(0) * (1 - away["is_dome"])
away["effective_temp"] = away["temp"].where(away["is_dome"] == 0, other=70).fillna(65)

vegas = pd.concat([home, away], ignore_index=True)[[
    "season", "week", "team",
    "implied_team_total", "is_home", "days_rest",
    "is_dome", "effective_wind", "effective_temp",
    "surface"
]]

print(vegas["is_home"].value_counts())
print(vegas.shape)
vegas.head(10)

is_home
1    1615
0    1615
Name: count, dtype: int64
(3230, 10)


,season,week,team,implied_team_total,is_home,days_rest,is_dome,effective_wind,effective_temp,surface
0,2020,1,KC,22.00,1,7,0,7.0,56.0,grass
1,2020,1,ATL,24.25,1,7,1,0.0,70.0,fieldturf
2,2020,1,BAL,20.00,1,7,0,5.0,76.0,grass
3,2020,1,BUF,16.50,1,7,0,15.0,67.0,astroturf
4,2020,1,CAR,25.50,1,7,0,5.0,81.0,grass
5,2020,1,DET,20.00,1,7,1,0.0,70.0,fieldturf
6,2020,1,JAX,25.50,1,7,0,3.0,80.0,grass
7,2020,1,MIN,22.00,1,7,1,0.0,70.0,sportturf
8,2020,1,NE,17.00,1,7,0,8.0,70.0,grass
9,2020,1,WAS,23.50,1,7,0,5.0,76.0,grass


In [80]:
# Fix dtypes so all key columns match before joining
for col in ["season", "week"]:
    player_stats_clean[col] = player_stats_clean[col].astype(int)
    ffo_clean[col] = ffo_clean[col].astype(int)
    vegas[col] = vegas[col].astype(int)

# Make sure game_id is string in both
player_stats_clean["game_id"] = player_stats_clean["game_id"].astype(str)
ffo_clean["game_id"] = ffo_clean["game_id"].astype(str)

# Make sure player_id is string in both
player_stats_clean["player_id"] = player_stats_clean["player_id"].astype(str)
ffo_clean["player_id"] = ffo_clean["player_id"].astype(str)

# Now join
df = player_stats_clean.copy()
df = df.merge(ffo_clean, on=["player_id", "season", "week", "game_id"], how="left")
df = df.merge(vegas, left_on=["team", "season", "week"],
                     right_on=["team", "season", "week"], how="left")

df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

# Fill ffo nulls — these are low usage players where expected pts ~ 0
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]
df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

print(df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

(34906, 50)
passing_epa      30939
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,...,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp,implied_team_total,is_home,days_rest,is_dome,effective_wind,effective_temp,surface
0,00-0019596,Tom Brady,QB,TB,NO,2020,1,None,20.46,20.46,...,0.0,0.0,0.0,26.25,0,7,1,0.0,70.0,astroturf
1,00-0019596,Tom Brady,QB,TB,CAR,2020,2,None,8.68,8.68,...,0.0,0.0,0.0,19.75,1,7,0,17.0,85.0,grass
2,00-0019596,Tom Brady,QB,TB,DEN,2020,3,None,23.88,23.88,...,0.0,0.0,0.0,18.25,0,7,0,7.0,55.0,grass
3,00-0019596,Tom Brady,QB,TB,LAC,2020,4,None,32.46,32.46,...,0.0,0.0,0.0,17.50,1,7,0,6.0,75.0,grass
4,00-0019596,Tom Brady,QB,TB,CHI,2020,5,None,14.12,14.12,...,0.0,0.0,0.0,20.25,0,4,0,7.0,57.0,grass


In [81]:
# Show all Justin Jefferson rows, no truncation
pd.set_option('display.max_rows', None)

sample = df[df["player_display_name"] == "Justin Jefferson"]
print(sample[["season", "week", "team", "targets", "receiving_yards",
              "fantasy_points", "ffo_expected_pts", "ffo_pts_diff",
              "implied_team_total", "is_home"]].to_string())

pd.reset_option('display.max_rows')

       season  week team  targets  receiving_yards  fantasy_points  ffo_expected_pts  ffo_pts_diff  implied_team_total  is_home
21018    2020     1  MIN        3               26            2.60              0.00          0.00               22.00        1
21019    2020     2  MIN        3               44            4.40              0.00          0.00               26.25        0
21020    2020     3  MIN        9              175           23.50              0.00          0.00               26.25        1
21021    2020     4  MIN        5              103           10.30              0.00          0.00               28.00        0
21022    2020     5  MIN        5               23            2.30              0.00          0.00               30.25        0
21023    2020     6  MIN       11              166           30.60              0.00          0.00               24.75        1
21024    2020     8  MIN        4               26            2.60              0.00          0.00      

In [82]:
# Check how many unique players are in each table
print("Unique players in player_stats:", player_stats_clean["player_id"].nunique())
print("Unique players in ffo:", ffo_clean["player_id"].nunique())

# Check overlap
ps_ids = set(player_stats_clean["player_id"].unique())
ffo_ids = set(ffo_clean["player_id"].unique())
print("Players in both:", len(ps_ids & ffo_ids))
print("Players only in player_stats (no ffo coverage):", len(ps_ids - ffo_ids))

Unique players in player_stats: 1240
Unique players in ffo: 1358
Players in both: 1190
Players only in player_stats (no ffo coverage): 50


In [83]:
# ffo coverage is partial — fill nulls with 0 for players with no opportunity data
# These are mostly low-usage players where expected pts ~ 0 anyway
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]

df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

# Verify no more nulls in these columns
print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Remaining nulls:
passing_epa      30939
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


## Part 3 — Injury, Depth Chart & Surface Features

In [84]:
# Filter to regular season only
injuries = nfl.load_injuries(SEASONS).to_pandas()

# Filter to regular season only
inj = injuries[injuries["game_type"] == "REG"].copy()

# Fix dtypes
inj["season"] = inj["season"].astype(int)
inj["week"] = inj["week"].astype(int)

# Encode report status
report_map = {
    None:           1.0,
    "Questionable": 0.5,
    "Doubtful":     0.25,
    "Out":          0.0,
    "Note":         1.0,
}
inj["injury_status_score"] = inj["report_status"].map(report_map).fillna(1.0)

# Encode practice status
practice_map = {
    "Full Participation in Practice":     1.0,
    "Limited Participation in Practice":  0.5,
    "Did Not Participate In Practice":    0.0,
    "\n":                                 1.0,
    "Note":                               1.0,
}
inj["practice_status_score"] = inj["practice_status"].map(practice_map).fillna(1.0)

# Keep ALL positions now — offensive and defensive
inj = inj[[
    "season", "week", "team", "gsis_id", "position",
    "full_name", "injury_status_score", "practice_status_score"
]].rename(columns={"gsis_id": "player_id"})

print(inj.shape)
print(inj["position"].value_counts().head(15))

(33400, 8)
position
WR    4458
LB    4252
CB    4074
T     2984
DT    2909
S     2640
DE    2578
RB    2448
TE    2144
G     2120
QB    1161
C      964
K      237
FB     173
P      134
Name: count, dtype: int64


### ~~Old starter proxy (cells 20-21) — REMOVED~~

> **Superseded.** This approach identified QB1 by highest season targets, which silently fails for QBs (0 targets). Replaced by depth-chart-based `teammate_flags` built in cells 22–28. The master rebuild cell (cell 34) uses those depth-chart flags. This cell is kept as a placeholder to avoid renumbering downstream cells.

> *(Continued from cell 20 — also superseded. The master rebuild cell 34 handles all df assembly cleanly.)*

In [87]:
# Filter to regular season, skill positions, starters only (depth_team = 1)
depth = nfl.load_depth_charts(SEASONS).to_pandas()

depth_clean = depth[
    (depth["game_type"] == "REG") &
    (depth["position"].isin(["QB", "RB", "WR", "TE"])) &
    (depth["depth_team"] == "1")  # string not int
].copy()

# Fix dtypes
depth_clean["season"] = depth_clean["season"].astype(int)
depth_clean["week"] = depth_clean["week"].astype(int)

# Keep relevant columns
depth_clean = depth_clean[[
    "season", "week", "club_code", "gsis_id", "position"
]].rename(columns={
    "club_code": "team",
    "gsis_id": "player_id"
})

# Sort so keep="last" picks the most recent entry per slot
depth_clean = depth_clean.sort_values(["season", "week", "team", "position", "player_id"]).drop_duplicates(subset=["season", "week", "team", "position"])

print(depth_clean.shape)
depth_clean.head(10)

(11961, 5)


,season,week,team,player_id,position
7,2020,1,ATL,00-0034837,WR
18,2020,1,ATL,00-0026143,QB
19,2020,1,ATL,00-0032241,RB
42,2020,1,ATL,00-0034830,TE
57,2020,2,ATL,00-0034837,WR
68,2020,2,ATL,00-0026143,QB
69,2020,2,ATL,00-0032241,RB
92,2020,2,ATL,00-0034830,TE
100,2020,3,ATL,00-0034830,TE
101,2020,3,ATL,00-0026143,QB


In [88]:
# All positions we want to track for opponent/teammate availability
# Defensive — affects opposing skill players
defensive_positions = ["CB", "OLB", "ILB", "MLB", "DE", "DT", "FS", "SS"]

# Offensive line — affects own team's skill players
ol_positions = ["T", "G", "C"]

# Get defensive starters from depth chart
def_depth = depth[
    (depth["game_type"] == "REG") &
    (depth["position"].isin(defensive_positions)) &
    (depth["depth_team"] == "1")
].copy()

def_depth["season"] = def_depth["season"].astype(int)
def_depth["week"] = def_depth["week"].astype(int)
def_depth = def_depth[[
    "season", "week", "club_code", "gsis_id", "position"
]].rename(columns={"club_code": "team", "gsis_id": "player_id"})
def_depth = def_depth.sort_values(["season", "week", "team", "position", "player_id"]).drop_duplicates(subset=["season", "week", "team", "position"])

# Get OL starters from depth chart
ol_depth = depth[
    (depth["game_type"] == "REG") &
    (depth["position"].isin(ol_positions)) &
    (depth["depth_team"] == "1")
].copy()

ol_depth["season"] = ol_depth["season"].astype(int)
ol_depth["week"] = ol_depth["week"].astype(int)
ol_depth = ol_depth[[
    "season", "week", "club_code", "gsis_id", "position"
]].rename(columns={"club_code": "team", "gsis_id": "player_id"})
ol_depth = ol_depth.sort_values(["season", "week", "team", "position", "player_id"]).drop_duplicates(subset=["season", "week", "team", "position"])

print("Defensive depth:", def_depth.shape)
print("OL depth:", ol_depth.shape)

Defensive depth: (18975, 5)
OL depth: (8766, 5)


In [89]:
# Join injury status onto defensive starters
def_starter_health = def_depth.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)
def_starter_health["injury_status_score"] = def_starter_health["injury_status_score"].fillna(1.0)
def_starter_health["practice_status_score"] = def_starter_health["practice_status_score"].fillna(1.0)
def_starter_health["def_starter_availability"] = (
    def_starter_health["injury_status_score"] * 0.6 +
    def_starter_health["practice_status_score"] * 0.4
)

# Join injury status onto OL starters
ol_starter_health = ol_depth.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)
ol_starter_health["injury_status_score"] = ol_starter_health["injury_status_score"].fillna(1.0)
ol_starter_health["practice_status_score"] = ol_starter_health["practice_status_score"].fillna(1.0)
ol_starter_health["ol_starter_availability"] = (
    ol_starter_health["injury_status_score"] * 0.6 +
    ol_starter_health["practice_status_score"] * 0.4
)

print("Defensive starter health:", def_starter_health.shape)
print("OL starter health:", ol_starter_health.shape)

Defensive starter health: (18975, 8)
OL starter health: (8766, 8)


In [90]:
# Pivot defensive flags — joined on opponent_team
def_flags = def_starter_health.pivot_table(
    index=["season", "week", "team"],
    columns="position",
    values="def_starter_availability",
    aggfunc="mean", observed=True
).reset_index()

def_flags.columns.name = None
def_flags = def_flags.rename(columns={
    "CB":  "opp_cb1_availability",
    "OLB": "opp_olb1_availability",
    "ILB": "opp_ilb1_availability",
    "MLB": "opp_mlb1_availability",
    "DE":  "opp_de1_availability",
    "DT":  "opp_dt1_availability",
    "FS":  "opp_fs1_availability",
    "SS":  "opp_ss1_availability",
})

def_flag_cols = ["opp_cb1_availability", "opp_olb1_availability", "opp_ilb1_availability",
                 "opp_mlb1_availability", "opp_de1_availability", "opp_dt1_availability",
                 "opp_fs1_availability", "opp_ss1_availability"]

for col in def_flag_cols:
    if col in def_flags.columns:
        def_flags[col] = def_flags[col].fillna(1.0)

# Pivot OL flags — joined on own team
ol_flags = ol_starter_health.pivot_table(
    index=["season", "week", "team"],
    columns="position",
    values="ol_starter_availability",
    aggfunc="mean", observed=True
).reset_index()

ol_flags.columns.name = None
ol_flags = ol_flags.rename(columns={
    "T": "starter_tackle_availability",
    "G": "starter_guard_availability",
    "C": "starter_center_availability",
})

ol_flag_cols = ["starter_tackle_availability", "starter_guard_availability",
                "starter_center_availability"]

for col in ol_flag_cols:
    if col in ol_flags.columns:
        ol_flags[col] = ol_flags[col].fillna(1.0)

print("Defensive flags:", def_flags.shape)
print("OL flags:", ol_flags.shape)

Defensive flags: (3008, 11)
OL flags: (3008, 6)


In [91]:
# Join defensive flags on opponent_team
df = df.merge(
    def_flags,
    left_on=["season", "week", "opponent_team"],
    right_on=["season", "week", "team"],
    how="left"
).drop(columns=["team_y"]).rename(columns={"team_x": "team"})

for col in def_flag_cols:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# Join OL flags on own team
df = df.merge(ol_flags, on=["season", "week", "team"], how="left")

for col in ol_flag_cols:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

print(df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])

(34907, 67)
passing_epa      30940
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


In [92]:
# Join injury status onto depth chart starters
starter_health = depth_clean.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)

# No injury report = healthy
starter_health["injury_status_score"] = starter_health["injury_status_score"].fillna(1.0)
starter_health["practice_status_score"] = starter_health["practice_status_score"].fillna(1.0)

# Combined availability score
starter_health["starter_availability"] = (
    starter_health["injury_status_score"] * 0.6 +
    starter_health["practice_status_score"] * 0.4
)

print(starter_health.shape)
starter_health.head(10)

(11963, 8)


,season,week,team,player_id,position,injury_status_score,practice_status_score,starter_availability
0,2020,1,ATL,00-0034837,WR,1.0,1.0,1.0
1,2020,1,ATL,00-0026143,QB,1.0,1.0,1.0
2,2020,1,ATL,00-0032241,RB,1.0,0.0,0.6
3,2020,1,ATL,00-0034830,TE,1.0,1.0,1.0
4,2020,2,ATL,00-0034837,WR,1.0,1.0,1.0
5,2020,2,ATL,00-0026143,QB,1.0,1.0,1.0
6,2020,2,ATL,00-0032241,RB,1.0,0.0,0.6
7,2020,2,ATL,00-0034830,TE,1.0,1.0,1.0
8,2020,3,ATL,00-0034830,TE,1.0,1.0,1.0
9,2020,3,ATL,00-0026143,QB,1.0,1.0,1.0


In [93]:
# Pivot to team level flags
teammate_flags = starter_health.pivot_table(
    index=["season", "week", "team"],
    columns="position",
    values="starter_availability",
    aggfunc="mean", observed=True
).reset_index()

teammate_flags.columns.name = None
teammate_flags = teammate_flags.rename(columns={
    "QB": "starter_qb_availability",
    "RB": "starter_rb_availability",
    "WR": "starter_wr_availability",
    "TE": "starter_te_availability",
})

# Fill missing — no report means healthy
for col in ["starter_qb_availability", "starter_rb_availability",
            "starter_wr_availability", "starter_te_availability"]:
    if col in teammate_flags.columns:
        teammate_flags[col] = teammate_flags[col].fillna(1.0)

print(teammate_flags.shape)
teammate_flags.head(10)

(3008, 7)


,season,week,team,starter_qb_availability,starter_rb_availability,starter_te_availability,starter_wr_availability
0,2020,1,ARI,1.0,1.0,1.0,1.0
1,2020,1,ATL,1.0,0.6,1.0,1.0
2,2020,1,BAL,1.0,0.0,1.0,1.0
3,2020,1,BUF,1.0,1.0,1.0,1.0
4,2020,1,CAR,1.0,1.0,1.0,1.0
5,2020,1,CHI,1.0,1.0,1.0,1.0
6,2020,1,CIN,1.0,1.0,1.0,1.0
7,2020,1,CLE,1.0,1.0,1.0,1.0
8,2020,1,DAL,1.0,1.0,1.0,1.0
9,2020,1,DEN,1.0,1.0,1.0,1.0


In [94]:
# Rebuild df cleanly from the base tables
df = player_stats_clean.copy()
df = df.merge(ffo_clean, on=["player_id", "season", "week", "game_id"], how="left")
df = df.drop_duplicates(subset=["player_id", "season", "week"], keep="first")
df = df.merge(vegas, left_on=["team", "season", "week"],
                     right_on=["team", "season", "week"], how="left")
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

# Fill ffo nulls
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]
df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

# Join player's own injury status
df = df.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)
df["injury_status_score"] = df["injury_status_score"].fillna(1.0)
df["practice_status_score"] = df["practice_status_score"].fillna(1.0)

# Join teammate availability flags
df = df.merge(teammate_flags, on=["season", "week", "team"], how="left")

for col in ["starter_qb_availability", "starter_rb_availability",
            "starter_wr_availability", "starter_te_availability"]:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum()[df.isnull().sum() > 0])

(34907, 56)
['player_id', 'player_display_name', 'position', 'team', 'opponent_team', 'season', 'week', 'game_id', 'fantasy_points', 'fantasy_points_ppr', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_air_yards', 'passing_epa', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost', 'rushing_epa', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_epa', 'target_share', 'air_yards_share', 'wopr', 'racr', 'ffo_actual_pts', 'ffo_expected_pts', 'ffo_pts_diff', 'rec_attempt', 'rush_attempt', 'rec_yards_gained_exp', 'rush_yards_gained_exp', 'rec_touchdown_exp', 'rush_touchdown_exp', 'implied_team_total', 'is_home', 'days_rest', 'is_dome', 'effective_wind', 'effective_temp', 'surface', 'injury_status_score', 'practice_status_score', 'starter_qb_availability', 'starter_rb_availability', 'starter_te_availability', 'starter_wr_availability

In [95]:
player_depth_rank = depth[
    (depth["game_type"] == "REG") &
    (depth["position"].isin(["QB", "RB", "WR", "TE"]))
].copy()

player_depth_rank["season"] = player_depth_rank["season"].astype(int)
player_depth_rank["week"] = player_depth_rank["week"].astype(int)
player_depth_rank["depth_team"] = player_depth_rank["depth_team"].astype(float).astype("Int64")

player_depth_rank = player_depth_rank[[
    "season", "week", "club_code", "gsis_id", "position", "depth_team"
]].rename(columns={
    "club_code": "team",
    "gsis_id": "player_id",
    "depth_team": "depth_chart_position"
})

player_depth_rank = (
    player_depth_rank
    .sort_values(["season", "week", "team", "position", "player_id"])
    .drop_duplicates(subset=["season", "week", "team", "player_id", "position"])
)

print(player_depth_rank.shape)
player_depth_rank.head()

# Rebuild df without game_id as a join key
df = player_stats_clean.copy()

# Join ffo using only player_id + season + week
df = df.merge(
    ffo_clean.drop(columns=["game_id"]),
    on=["player_id", "season", "week"],
    how="left"
)

df = df.merge(vegas, left_on=["team", "season", "week"],
                     right_on=["team", "season", "week"], how="left")

df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

# Fill ffo nulls
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]
df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

# Join injury status
df = df.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)
df["injury_status_score"] = df["injury_status_score"].fillna(1.0)
df["practice_status_score"] = df["practice_status_score"].fillna(1.0)

# Join teammate availability flags
df = df.merge(teammate_flags, on=["season", "week", "team"], how="left")
for col in ["starter_qb_availability", "starter_rb_availability",
            "starter_wr_availability", "starter_te_availability"]:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# Join depth chart position
df = df.merge(player_depth_rank, on=["season", "week", "team", "player_id", "position"], how="left")
df["depth_chart_position"] = df["depth_chart_position"].fillna(3)

print(df.shape)
print("ffo_expected_pts > 0:", (df["ffo_expected_pts"] > 0).sum())
print(df.isnull().sum()[df.isnull().sum() > 0])

(43458, 6)
(34907, 57)
ffo_expected_pts > 0: 31342
passing_epa      30940
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


In [96]:
# Clean surface values
df["surface"] = df["surface"].str.strip().str.lower()

# Simplify to binary — grass vs turf
# Research shows turf has higher injury rates and slightly different play styles
df["is_turf"] = df["surface"].isin([
    "fieldturf", "matrixturf", "sportturf", "astroturf", "a_turf"
]).astype(int)

# Fill blanks — most missing are older games, default to grass (more common)
df["is_turf"] = df["is_turf"].fillna(0).astype(int)

# Drop original surface string column
df = df.drop(columns=["surface"])

print(df["is_turf"].value_counts())
print(df.shape)

is_turf
0    19512
1    15395
Name: count, dtype: int64
(34907, 57)


In [97]:
# Check if defensive players appear in the raw injuries table
print("All positions in raw injuries table:")
print(injuries["position"].value_counts().head(30))

All positions in raw injuries table:
position
WR        4650
LB        4438
CB        4245
T         3109
DT        3013
S         2743
DE        2680
RB        2560
TE        2230
G         2211
QB        1232
C         1004
K          250
FB         179
P          138
LS         118
\n          12
Name: count, dtype: int64


## Master Rebuild

Run this single cell to reconstruct `df` cleanly from all base tables. Execute this before running any Part 4 step.

In [98]:
# ── Master rebuild — run this single cell to rebuild df through Part 3 (player stats, FFO, schedule, injuries, depth charts, team metrics). Then continue running Part 4 cells for coach win%, offensive/defensive metrics, and AllPro counts. ──────────────────

# Start fresh from base tables
df = player_stats_clean.copy()

# 1. Join ffo
df = df.merge(
    ffo_clean.drop(columns=["game_id"]),
    on=["player_id", "season", "week"],
    how="left"
)
df = df.drop_duplicates(subset=["player_id", "season", "week"], keep="first")

# 2. Join Vegas + weather
df = df.merge(vegas, left_on=["team", "season", "week"],
                     right_on=["team", "season", "week"], how="left")

# Fill ffo nulls
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]
df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

# 3. Join player's own injury status
df = df.merge(
    inj[["season", "week", "player_id", "injury_status_score", "practice_status_score"]],
    on=["season", "week", "player_id"],
    how="left"
)
df["injury_status_score"] = df["injury_status_score"].fillna(1.0)
df["practice_status_score"] = df["practice_status_score"].fillna(1.0)

# 4. Join offensive teammate availability
df = df.merge(teammate_flags, on=["season", "week", "team"], how="left")
for col in ["starter_qb_availability", "starter_rb_availability",
            "starter_wr_availability", "starter_te_availability"]:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# 5. Join depth chart position
df = df.merge(player_depth_rank, on=["season", "week", "team", "player_id", "position"], how="left")
df["depth_chart_position"] = df["depth_chart_position"].fillna(3)

# 6. Encode surface
df["surface"] = df["surface"].str.strip().str.lower()
df["is_turf"] = df["surface"].isin([
    "fieldturf", "matrixturf", "sportturf", "astroturf", "a_turf"
]).astype(int)
df = df.drop(columns=["surface"])

# 7. Join defensive flags on opponent_team
df = df.merge(
    def_flags,
    left_on=["season", "week", "opponent_team"],
    right_on=["season", "week", "team"],
    how="left"
).drop(columns=["team_y"]).rename(columns={"team_x": "team"})

for col in def_flag_cols:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# 8. Join OL flags on own team
df = df.merge(ol_flags, on=["season", "week", "team"], how="left")
for col in ol_flag_cols:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# Sort
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

print(df.shape)
print("New columns:", [c for c in df.columns if "opp_" in c or "starter_tackle" in c
                       or "starter_guard" in c or "starter_center" in c])
print(df.isnull().sum()[df.isnull().sum() > 0])

(34907, 68)
New columns: ['opp_cb1_availability', 'opp_de1_availability', 'opp_dt1_availability', 'opp_fs1_availability', 'opp_ilb1_availability', 'opp_mlb1_availability', 'opp_olb1_availability', 'opp_ss1_availability', 'starter_center_availability', 'starter_guard_availability', 'starter_tackle_availability']
passing_epa      30940
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


## Part 4 — New Feature Groups

### Step 1 — Coach Win Percentage

In [99]:
# Load schedules from 1999 onward for full coaching career history
COACH_SEASONS = list(range(1999, 2026))
raw_coach = nfl.load_schedules(COACH_SEASONS).to_pandas()
coach_hist = raw_coach[
    (raw_coach["game_type"] == "REG") &
    (raw_coach["result"].notna())
].copy()
coach_hist["season"] = coach_hist["season"].astype(int)
coach_hist["week"]   = coach_hist["week"].astype(int)

# Normalize legacy team abbreviations
TEAM_MAP = {
    "STL": "LA",  "LAR": "LA",  "OAK": "LV",  "LVR": "LV",
    "SD":  "LAC", "SDG": "LAC", "NWE": "NE",  "KAN": "KC",
    "GNB": "GB",  "NOR": "NO",  "TAM": "TB",  "SFO": "SF",
}
coach_hist["home_team"] = coach_hist["home_team"].replace(TEAM_MAP)
coach_hist["away_team"] = coach_hist["away_team"].replace(TEAM_MAP)

# Long format - one row per team per game
home_c = coach_hist[["season", "week", "home_team", "home_score", "away_score", "home_coach"]].copy()
home_c.rename(columns={"home_team": "team", "home_score": "team_score", "away_score": "opp_score", "home_coach": "coach"}, inplace=True)

away_c = coach_hist[["season", "week", "away_team", "away_score", "home_score", "away_coach"]].copy()
away_c.rename(columns={"away_team": "team", "away_score": "team_score", "home_score": "opp_score", "away_coach": "coach"}, inplace=True)

games_df = pd.concat([home_c, away_c], ignore_index=True)
games_df["win"] = (games_df["team_score"] > games_df["opp_score"]).astype(int)
games_df = games_df.sort_values(["coach", "season", "week"]).reset_index(drop=True)

# Cumulative win% BEFORE this game - per-group transform avoids leakage across coaches
games_df["cumulative_wins"]  = games_df.groupby("coach")["win"].transform(
    lambda x: x.shift(1, fill_value=0).cumsum()
)
games_df["cumulative_games"] = games_df.groupby("coach").cumcount()  # 0-indexed = games before this one

# NaN for coaches with < 10 prior games - too noisy to be useful
games_df["coach_win_pct"] = (
    games_df["cumulative_wins"] / games_df["cumulative_games"]
).where(games_df["cumulative_games"] >= 10)
games_df["coach_win_pct"] = games_df["coach_win_pct"].round(4)

# Lookup table scoped to our model seasons
coach_lkp = games_df[games_df["season"].isin(SEASONS)][["season", "week", "team", "coach_win_pct"]].copy()

# Drop if re-running this cell so the merge does not create _x/_y suffixes
df = df.drop(columns=["coach_win_pct", "opp_coach_win_pct"], errors="ignore")

# Join for player's own team coach
df = df.merge(coach_lkp, on=["season", "week", "team"], how="left")

# Join for opponent's coach
df = df.merge(
    coach_lkp.rename(columns={"team": "opponent_team", "coach_win_pct": "opp_coach_win_pct"}),
    on=["season", "week", "opponent_team"],
    how="left"
)

print(f"Shape: {df.shape}")
print(f"coach_win_pct     - nulls: {df['coach_win_pct'].isna().sum()}, "
      f"mean: {df['coach_win_pct'].mean():.3f}, "
      f"min: {df['coach_win_pct'].min():.3f}, "
      f"max: {df['coach_win_pct'].max():.3f}")
print(f"opp_coach_win_pct - nulls: {df['opp_coach_win_pct'].isna().sum()}")
print()
print("Sample - Patrick Mahomes:")
print(df[df["player_display_name"] == "Patrick Mahomes"][
    ["season", "week", "team", "opponent_team", "coach_win_pct", "opp_coach_win_pct"]
].head(10).to_string())


Shape: (34907, 70)
coach_win_pct     - nulls: 3435, mean: 0.525, min: 0.000, max: 0.850
opp_coach_win_pct - nulls: 3444

Sample - Patrick Mahomes:
      season  week team opponent_team  coach_win_pct  opp_coach_win_pct
9625    2020     1   KC           HOU         0.6161             0.5417
9626    2020     2   KC           LAC         0.6172             0.5400
9627    2020     3   KC           BAL         0.6183             0.6186
9628    2020     4   KC            NE         0.6195             0.7399
9629    2020     5   KC            LV         0.6206             0.5102
9630    2020     6   KC           BUF         0.6188             0.5472
9631    2020     7   KC           DEN         0.6199             0.4286
9632    2020     8   KC           NYJ         0.6210             0.4225
9633    2020     9   KC           CAR         0.6221                NaN
9634    2020    11   KC            LV         0.6232             0.5174


### Step 2 — Offensive Team Metrics

In [100]:
# Load PBP data (this takes ~60s)
print("Loading PBP data...")
pbp_raw = nfl.load_pbp(SEASONS).to_pandas()
print(f"PBP loaded: {pbp_raw.shape}")

# Filter to run/pass plays with valid possession team
pbp = pbp_raw[
    pbp_raw["play_type"].isin(["run", "pass"]) &
    pbp_raw["posteam"].notna()
].copy()
pbp["season"] = pbp["season"].astype(int)
pbp["week"]   = pbp["week"].astype(int)

# Per-game offensive aggregations
pbp["is_pass"]  = (pbp["play_type"] == "pass").astype(int)
pbp["is_rz"]    = (pbp["yardline_100"] <= 20).astype(int)
pbp["is_rz_td"] = ((pbp["yardline_100"] <= 20) & (pbp["touchdown"] == 1)).astype(int)

off_game = pbp.groupby(["season", "week", "posteam"]).agg(
    epa_sum    =("epa",          "sum"),
    yards_sum  =("yards_gained", "sum"),
    play_count =("play_id",      "count"),
    pass_count =("is_pass",      "sum"),
    rz_plays   =("is_rz",        "sum"),
    rz_tds     =("is_rz_td",     "sum"),
).reset_index().rename(columns={"posteam": "team"})

off_game["epa_per_play"]   = off_game["epa_sum"]   / off_game["play_count"]
off_game["yards_per_play"] = off_game["yards_sum"]  / off_game["play_count"]
off_game["pass_rate"]      = off_game["pass_count"] / off_game["play_count"]
off_game["rz_score_rate"]  = (
    off_game["rz_tds"] / off_game["rz_plays"].replace(0, float("nan"))
)

off_game = off_game.sort_values(["team", "season", "week"]).reset_index(drop=True)

# Rolling 4-week averages, shift(1) to avoid leakage
ROLL4 = {
    "epa_per_play":   "off_epa_roll4",
    "yards_per_play": "off_yards_per_play_roll4",
    "pass_rate":      "off_pass_rate_roll4",
    "rz_score_rate":  "off_red_zone_rate_roll4",
}
for src, dst in ROLL4.items():
    off_game[dst] = off_game.groupby(["team", "season"])[src].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

roll_cols = list(ROLL4.values())
off_lkp = off_game[["season", "week", "team"] + roll_cols].copy()

# Drop if re-running this cell
df = df.drop(columns=roll_cols, errors="ignore")

# Join onto df using player's own team
df = df.merge(off_lkp, on=["season", "week", "team"], how="left")

print(f"Shape: {df.shape}")
for col in roll_cols:
    print(f"  {col:<30} — nulls: {df[col].isna().sum()}, mean: {df[col].mean():.4f}")

print()
print("Sample — Patrick Mahomes (KC offense):")
print(df[df["player_display_name"] == "Patrick Mahomes"][
    ["season", "week", "team"] + roll_cols
].head(8).to_string())


Loading PBP data...
PBP loaded: (294989, 372)
Shape: (34907, 74)
  off_epa_roll4                  — nulls: 349, mean: -0.0030
  off_yards_per_play_roll4       — nulls: 349, mean: 5.4739
  off_pass_rate_roll4            — nulls: 349, mean: 0.5799
  off_red_zone_rate_roll4        — nulls: 349, mean: 0.2166

Sample — Patrick Mahomes (KC offense):
      season  week team  off_epa_roll4  off_yards_per_play_roll4  off_pass_rate_roll4  off_red_zone_rate_roll4
9625    2020     1   KC            NaN                       NaN                  NaN                      NaN
9626    2020     2   KC       0.213770                  5.507463             0.492537                 0.166667
9627    2020     3   KC       0.165973                  5.683309             0.591339                 0.208333
9628    2020     4   KC       0.217981                  6.149603             0.586007                 0.287037
9629    2020     5   KC       0.177214                  6.193933             0.583736              

### Step 3 — Defensive Team Metrics

In [101]:
# Reuse pbp from Step 2 (already loaded and filtered)
# Per-game defensive aggregations — grouped by defteam
def_game = pbp.groupby(["season", "week", "defteam"]).agg(
    epa_sum    =("epa",          "sum"),
    yards_sum  =("yards_gained", "sum"),
    play_count =("play_id",      "count"),
    pass_count =("is_pass",      "sum"),
    rz_plays   =("is_rz",        "sum"),
    rz_tds     =("is_rz_td",     "sum"),
).reset_index().rename(columns={"defteam": "team"})

def_game["epa_allowed_per_play"]  = def_game["epa_sum"]   / def_game["play_count"]
def_game["yards_allowed_per_play"]= def_game["yards_sum"]  / def_game["play_count"]
def_game["pass_rate_faced"]       = def_game["pass_count"] / def_game["play_count"]
def_game["rz_allowed_rate"]       = (
    def_game["rz_tds"] / def_game["rz_plays"].replace(0, float("nan"))
)

def_game = def_game.sort_values(["team", "season", "week"]).reset_index(drop=True)

# Rolling 4-week averages, shift(1) to avoid leakage
DEF_ROLL4 = {
    "epa_allowed_per_play":   "def_epa_allowed_roll4",
    "yards_allowed_per_play": "def_yards_allowed_roll4",
    "pass_rate_faced":        "def_pass_rate_faced_roll4",
    "rz_allowed_rate":        "def_red_zone_allowed_roll4",
}
for src, dst in DEF_ROLL4.items():
    def_game[dst] = def_game.groupby(["team", "season"])[src].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

def_roll_cols = list(DEF_ROLL4.values())
def_lkp = def_game[["season", "week", "team"] + def_roll_cols].copy()

# Drop if re-running this cell
df = df.drop(columns=def_roll_cols, errors="ignore")

# Join onto df using player's opponent_team
df = df.merge(
    def_lkp.rename(columns={"team": "opponent_team"}),
    on=["season", "week", "opponent_team"],
    how="left"
)

print(f"Shape: {df.shape}")
for col in def_roll_cols:
    print(f"  {col:<30} — nulls: {df[col].isna().sum()}, mean: {df[col].mean():.4f}")

print()
print("Sample — Patrick Mahomes (opposing defense):")
print(df[df["player_display_name"] == "Patrick Mahomes"][
    ["season", "week", "opponent_team"] + def_roll_cols
].head(8).to_string())


Shape: (34907, 78)
  def_epa_allowed_roll4          — nulls: 349, mean: 0.0035
  def_yards_allowed_roll4        — nulls: 349, mean: 5.5020
  def_pass_rate_faced_roll4      — nulls: 349, mean: 0.5780
  def_red_zone_allowed_roll4     — nulls: 349, mean: 0.2174

Sample — Patrick Mahomes (opposing defense):
      season  week opponent_team  def_epa_allowed_roll4  def_yards_allowed_roll4  def_pass_rate_faced_roll4  def_red_zone_allowed_roll4
9625    2020     1           HOU                    NaN                      NaN                        NaN                         NaN
9626    2020     2           LAC              -0.216665                 4.469697                   0.575758                    0.000000
9627    2020     3           BAL              -0.225093                 5.005330                   0.654184                    0.416667
9628    2020     4            NE              -0.010037                 6.178695                   0.547209                    0.205051
9629    2020   

### Step 4 — All Pro Stats

### Step 4b — Join All-Pro Data

> **Note:** `ALLPRO_PATH` auto-detects CWD — works whether papermill is run from `fantasy/` or the project root.

In [102]:

import pathlib as _pl
_allpro_candidates = [
    _pl.Path("../betting/nfl_allpro_1997_2025.csv"),   # CWD = fantasy/
    _pl.Path("betting/nfl_allpro_1997_2025.csv"),      # CWD = project root
]
ALLPRO_PATH = next(p for p in _allpro_candidates if p.exists())
allpro = pd.read_csv(ALLPRO_PATH)

# Normalize legacy abbreviations
ALLPRO_TEAM_MAP = {
    "STL": "LA",  "LAR": "LA",  "OAK": "LV",  "LVR": "LV",
    "SD":  "LAC", "SDG": "LAC", "NWE": "NE",  "KAN": "KC",
    "GNB": "GB",  "NOR": "NO",  "TAM": "TB",  "SFO": "SF",
}
allpro["Team"] = allpro["Team"].replace(ALLPRO_TEAM_MAP)
allpro = allpro[allpro["Team"] != "2TM"].copy()

offense_ap = allpro[allpro["Side"] == "offense"].copy()
defense_ap = allpro[allpro["Side"] == "defense"].copy()

def build_weighted(df_ap, seasons):
    """Weighted 3-year lookback: 1yr-back=4pts, 2yr=2pts, 3yr=1pt.
    Players in multiple lookback years get only the highest weight.
    """
    frames = []
    for season in seasons:
        curr = []
        for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
            tmp = df_ap[df_ap["Year"] == season - yrs_back].copy()
            tmp["weight"] = weight
            tmp["season"] = season
            curr.append(tmp)
        comb    = pd.concat(curr)
        deduped = comb.sort_values("weight", ascending=False).drop_duplicates(["Player", "season"])
        wc      = deduped.groupby(["season", "Team"])["weight"].sum().reset_index()
        wc.columns = ["season", "Team", "allpro_weighted"]
        frames.append(wc)
    return pd.concat(frames, ignore_index=True)

weighted_all     = build_weighted(allpro,     SEASONS)
weighted_offense = build_weighted(offense_ap, SEASONS)
weighted_defense = build_weighted(defense_ap, SEASONS)

NEW_COLS = [
    "team_allpro_weighted", "team_offense_allpro", "team_defense_allpro",
    "opp_allpro_weighted",  "opp_offense_allpro",  "opp_defense_allpro",
]
df = df.drop(columns=NEW_COLS, errors="ignore")

# Own team joins
df = df.merge(
    weighted_all.rename(columns={"Team": "team", "allpro_weighted": "team_allpro_weighted"}),
    on=["season", "team"], how="left"
)
df = df.merge(
    weighted_offense.rename(columns={"Team": "team", "allpro_weighted": "team_offense_allpro"}),
    on=["season", "team"], how="left"
)
df = df.merge(
    weighted_defense.rename(columns={"Team": "team", "allpro_weighted": "team_defense_allpro"}),
    on=["season", "team"], how="left"
)

# Opponent joins
df = df.merge(
    weighted_all.rename(columns={"Team": "opponent_team", "allpro_weighted": "opp_allpro_weighted"}),
    on=["season", "opponent_team"], how="left"
)
df = df.merge(
    weighted_offense.rename(columns={"Team": "opponent_team", "allpro_weighted": "opp_offense_allpro"}),
    on=["season", "opponent_team"], how="left"
)
df = df.merge(
    weighted_defense.rename(columns={"Team": "opponent_team", "allpro_weighted": "opp_defense_allpro"}),
    on=["season", "opponent_team"], how="left"
)

# Teams with no All-Pros get 0
for col in NEW_COLS:
    df[col] = df[col].fillna(0)

print(f"Shape: {df.shape}")
for col in NEW_COLS:
    print(f"  {col:<28} - nulls: {df[col].isna().sum()}, mean: {df[col].mean():.3f}, max: {df[col].max():.0f}")

print()
print("Top 5 teams by team_allpro_weighted (2024):")
top5 = weighted_all[weighted_all["season"] == 2024].sort_values("allpro_weighted", ascending=False).head(5)
print(top5.to_string(index=False))

print()
print("Sample - Patrick Mahomes:")
print(df[df["player_display_name"] == "Patrick Mahomes"][
    ["season", "week", "team"] + NEW_COLS
].head(6).to_string())


Shape: (34907, 84)
  team_allpro_weighted         - nulls: 0, mean: 12.397, max: 43
  team_offense_allpro          - nulls: 0, mean: 5.867, max: 24
  team_defense_allpro          - nulls: 0, mean: 6.530, max: 25
  opp_allpro_weighted          - nulls: 0, mean: 12.359, max: 43
  opp_offense_allpro           - nulls: 0, mean: 5.833, max: 24
  opp_defense_allpro           - nulls: 0, mean: 6.526, max: 25

Top 5 teams by team_allpro_weighted (2024):
 season Team  allpro_weighted
   2024   SF               35
   2024  DAL               33
   2024  PHI               30
   2024  BAL               25
   2024   KC               22

Sample - Patrick Mahomes:
      season  week team  team_allpro_weighted  team_offense_allpro  team_defense_allpro  opp_allpro_weighted  opp_offense_allpro  opp_defense_allpro
9625    2020     1   KC                  21.0                 13.0                  8.0                 17.0                 8.0                 9.0
9626    2020     2   KC                  21.0

### Step 5 — Snap Share

Weekly offensive snap percentage from Pro Football Reference via `nfl.load_snap_counts()`. Joined to `player_id` through the PFR→GSIS bridge in `load_players()`.

Added columns: `snap_pct_roll3`, `snap_pct_roll5`, `snap_pct_trend` (roll3 − roll5)

In [ ]:
print("Loading snap counts...")
_players_snap = nfl.load_players().to_pandas()
_snap_bridge = (
    _players_snap.loc[
        _players_snap['pfr_id'].notna() & _players_snap['gsis_id'].notna(),
        ['pfr_id', 'gsis_id']
    ].drop_duplicates()
)

snap_raw = nfl.load_snap_counts(SEASONS).to_pandas()
snap = (
    snap_raw[snap_raw['game_type'] == 'REG']
    [['pfr_player_id', 'season', 'week', 'offense_pct']]
    .merge(_snap_bridge, left_on='pfr_player_id', right_on='pfr_id', how='left')
    .rename(columns={'gsis_id': 'player_id'})
    [['player_id', 'season', 'week', 'offense_pct']]
    .dropna(subset=['player_id'])
)
snap['season'] = snap['season'].astype(int)
snap['week']   = snap['week'].astype(int)

snap = snap.sort_values(['player_id', 'season', 'week'])
snap['snap_pct_roll3'] = (
    snap.groupby(['player_id', 'season'])['offense_pct']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
snap['snap_pct_roll5'] = (
    snap.groupby(['player_id', 'season'])['offense_pct']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
snap['snap_pct_trend'] = snap['snap_pct_roll3'] - snap['snap_pct_roll5']

df = df.merge(
    snap[['player_id', 'season', 'week',
          'snap_pct_roll3', 'snap_pct_roll5', 'snap_pct_trend']],
    on=['player_id', 'season', 'week'], how='left'
)
df[['snap_pct_roll3', 'snap_pct_roll5', 'snap_pct_trend']] = (
    df[['snap_pct_roll3', 'snap_pct_roll5', 'snap_pct_trend']].fillna(0)
)
pct_covered = (snap_raw['game_type'] == 'REG').mean()
print(f"snap_pct added | null={df['snap_pct_roll3'].isna().sum()} "
      f"| non-zero={( df['snap_pct_roll3'] > 0).sum()}")


### Save Raw Dataset

Writes `fantasy/raw_dataset.csv`. This is the input for `features.ipynb`. Do not run `features.ipynb` or `model.ipynb` until this cell completes.

In [103]:
# Save the joined dataset
df.to_csv("raw_dataset.csv", index=False)
print(f"Saved: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Null counts for all new feature columns:")
new_cols = [
    "coach_win_pct", "opp_coach_win_pct",
    "off_epa_roll4", "off_yards_per_play_roll4", "off_pass_rate_roll4", "off_red_zone_rate_roll4",
    "def_epa_allowed_roll4", "def_yards_allowed_roll4", "def_pass_rate_faced_roll4", "def_red_zone_allowed_roll4",
    "team_allpro_weighted", "team_offense_allpro", "team_defense_allpro",
    "opp_allpro_weighted",  "opp_offense_allpro",  "opp_defense_allpro",
]
for col in new_cols:
    if col in df.columns:
        nulls = df[col].isna().sum()
        pct   = nulls / len(df) * 100
        print(f"  {col:<34} nulls: {nulls:>5} ({pct:.1f}%)")


Saved: 34907 rows x 84 columns

Null counts for all new feature columns:
  coach_win_pct                      nulls:  3435 (9.8%)
  opp_coach_win_pct                  nulls:  3444 (9.9%)
  off_epa_roll4                      nulls:   349 (1.0%)
  off_yards_per_play_roll4           nulls:   349 (1.0%)
  off_pass_rate_roll4                nulls:   349 (1.0%)
  off_red_zone_rate_roll4            nulls:   349 (1.0%)
  def_epa_allowed_roll4              nulls:   349 (1.0%)
  def_yards_allowed_roll4            nulls:   349 (1.0%)
  def_pass_rate_faced_roll4          nulls:   349 (1.0%)
  def_red_zone_allowed_roll4         nulls:   349 (1.0%)
  team_allpro_weighted               nulls:     0 (0.0%)
  team_offense_allpro                nulls:     0 (0.0%)
  team_defense_allpro                nulls:     0 (0.0%)
  opp_allpro_weighted                nulls:     0 (0.0%)
  opp_offense_allpro                 nulls:     0 (0.0%)
  opp_defense_allpro                 nulls:     0 (0.0%)


## Validation & Spot Checks

In [104]:
pd.set_option('display.max_columns', None)
df

,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_air_yards,passing_epa,carries,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_epa,target_share,air_yards_share,wopr,racr,ffo_actual_pts,ffo_expected_pts,ffo_pts_diff,rec_attempt,rush_attempt,rec_yards_gained_exp,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp,implied_team_total,is_home,days_rest,is_dome,effective_wind,effective_temp,injury_status_score,practice_status_score,starter_qb_availability,starter_rb_availability,starter_te_availability,starter_wr_availability,depth_chart_position,is_turf,opp_cb1_availability,opp_de1_availability,opp_dt1_availability,opp_fs1_availability,opp_ilb1_availability,opp_mlb1_availability,opp_olb1_availability,opp_ss1_availability,starter_center_availability,starter_guard_availability,starter_tackle_availability,coach_win_pct,opp_coach_win_pct,off_epa_roll4,off_yards_per_play_roll4,off_pass_rate_roll4,off_red_zone_rate_roll4,def_epa_allowed_roll4,def_yards_allowed_roll4,def_pass_rate_faced_roll4,def_red_zone_allowed_roll4,team_allpro_weighted,team_offense_allpro,team_defense_allpro,opp_allpro_weighted,opp_offense_allpro,opp_defense_allpro
0,00-0019596,Tom Brady,QB,TB,NO,2020,1,None,20.46,20.46,23,36,239,2,2,292,-10.443857,3,9,1,0,1.505448,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,20.46,19.90,0.56,0.0,3.0,0.00,13.23,0.00,0.57,26.25,0,7,1,0.0,70.0,1.0,1.0,1.0,1.0,1.0,1.00,1,1,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.5833,0.6298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,13.0,8.0,30.0,14.0,16.0
1,00-0019596,Tom Brady,QB,TB,CAR,2020,2,None,8.68,8.68,23,35,217,1,1,234,0.524379,1,0,0,1,-5.488591,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,8.68,16.87,-8.19,0.0,1.0,0.00,0.00,0.00,0.00,19.75,1,7,0,17.0,85.0,1.0,1.0,1.0,1.0,1.0,0.35,1,0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.5773,NaN,-0.156884,4.696970,0.606061,0.230769,0.247142,6.098361,0.491803,0.272727,21.0,13.0,8.0,11.0,6.0,5.0
2,00-0019596,Tom Brady,QB,TB,DEN,2020,3,None,23.88,23.88,25,38,297,3,0,311,11.559702,5,0,0,0,-3.811726,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,23.88,23.01,0.87,0.0,5.0,0.00,4.63,0.00,0.05,18.25,0,7,0,7.0,55.0,1.0,1.0,1.0,1.0,1.0,1.00,1,0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,1.0,1.0,1.0,1.0,0.5816,0.3889,-0.088305,5.322169,0.610048,0.226496,-0.000869,5.739247,0.620761,0.132479,21.0,13.0,8.0,8.0,2.0,6.0
3,00-0019596,Tom Brady,QB,TB,LAC,2020,4,None,32.46,32.46,30,46,369,5,1,434,12.685636,3,-3,0,0,-1.166074,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,32.46,25.04,7.42,0.0,3.0,0.00,-3.00,0.00,0.00,17.50,1,7,0,6.0,75.0,1.0,1.0,1.0,1.0,1.0,1.00,1,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.7,1.0,1.0,1.0,0.5859,0.5192,0.004331,5.451339,0.621752,0.213497,-0.066521,5.416807,0.618045,0.107143,21.0,13.0,8.0,21.0,9.0,12.0
4,00-0019596,Tom Brady,QB,TB,CHI,2020,5,None,14.12,14.12,25,41,253,1,0,383,0.327568,3,0,0,0,1.146621,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,14.12,18.45,-4.33,0.0,3.0,0.00,1.24,0.00,0.00,20.25,0,4,0,7.0,57.0,1.0,1.0,1.0,1.0,0.6,1.00,1,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5900,0.6389,0.046757,5.827790,0.630600,0.243456,-0.036838,5.309215,0.594522,0.136080,21.0,13.0,8.0,9.0,0.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34902,00-0040784,Quinshon Judkins,RB,CLE,LV,2025,12,2025_12_CLE_LV,16.70,16.70,0,0,0,0,0,0,NaN,16,47,2,0,-1.999942,0,0,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN,16.70,10.42,6.28,0.0,16.0,0.00,62.51,0.00,0.69,19.25,0,7,1,0.0,70.0,1.0,1.0,1.0,1.0

In [105]:
# Find a game where opp_cb1_availability is actually low (injured CB)
print("=== Games where opposing CB1 was injured ===")
print(df[
    (df["opp_cb1_availability"] < 0.5) &
    (df["position"] == "WR") &
    (df["season"] == 2021)
][["player_display_name", "team", "opponent_team", "week", "targets",
   "receiving_yards", "opp_cb1_availability"]].head(15).to_string())

# Sanity check 2 — OL injury check
print("=== OL Availability < 0.5 vs RB Production ===")
print(df[
    (df["position"] == "RB") &
    (df["season"] == 2022) &
    (df["starter_tackle_availability"] < 0.5)
][["player_display_name", "team", "week", "carries", "rushing_yards",
   "starter_tackle_availability", "starter_guard_availability"]].head(15).to_string())

# Sanity check 3 — distribution check
print("=== Defensive Flag Distributions ===")
for col in def_flag_cols:
    if col in df.columns:
        print(f"{col}: {df[col].value_counts().head(4).to_dict()}")

print("\n=== OL Flag Distributions ===")
for col in ol_flag_cols:
    if col in df.columns:
        print(f"{col}: {df[col].value_counts().head(4).to_dict()}")

=== Games where opposing CB1 was injured ===
     player_display_name team opponent_team  week  targets  receiving_yards  opp_cb1_availability
349       Danny Amendola  HOU           TEN    11        1                0                  0.30
450       DeSean Jackson   LV           CLE    15        3               11                  0.00
758        Andre Roberts  LAC           PIT    11        0                0                  0.00
886           A.J. Green  ARI           SEA    18        9               23                  0.00
914          Julio Jones  TEN            LA     9        4               35                  0.30
1266        Cole Beasley  BUF           ATL    17        6               22                  0.15
1440        Marvin Jones  JAX           TEN    14        7               70                  0.00
1610         Josh Gordon   KC           DEN    18        1                0                  0.00
1708        Adam Thielen  MIN           LAC    10        7               

In [106]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Pick a well known QB with a long career in our dataset
sample = df[df["player_display_name"] == "Patrick Mahomes"].head(20)
print(sample[[
    # identity
    "season", "week", "team", "opponent_team",
    # raw stats
    "completions", "attempts", "passing_yards", "passing_tds", "passing_interceptions",
    # fantasy points
    "fantasy_points", "fantasy_points_ppr",
    # ffo
    "ffo_expected_pts", "ffo_pts_diff",
    # vegas
    "implied_team_total", "is_home",
    # weather
    "effective_wind", "effective_temp", "is_dome", "is_turf",
    # rest
    "days_rest",
    # injury
    "injury_status_score", "practice_status_score",
    # teammate availability
    "starter_qb_availability", "starter_rb_availability",
    "starter_wr_availability", "starter_te_availability",
    # depth chart
    "depth_chart_position"
]].to_string())

pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')

      season  week team opponent_team  completions  attempts  passing_yards  passing_tds  passing_interceptions  fantasy_points  fantasy_points_ppr  ffo_expected_pts  ffo_pts_diff  implied_team_total  is_home  effective_wind  effective_temp  is_dome  is_turf  days_rest  injury_status_score  practice_status_score  starter_qb_availability  starter_rb_availability  starter_wr_availability  starter_te_availability  depth_chart_position
9625    2020     1   KC           HOU           24        32            211            3                      0           20.44               20.44             20.74         -0.30               22.00        1             7.0            56.0        0        0          7                  1.0                    1.0                      1.0                      1.0                      1.0                      1.0                     1
9626    2020     2   KC           LAC           27        47            302            2                      0           27.48 

In [107]:
# Check a WR — Davante Adams
sample_wr = df[df["player_display_name"] == "Davante Adams"].head(10)
print(sample_wr[[
    "season", "week", "team", "targets", "receptions", "receiving_yards",
    "receiving_tds", "fantasy_points", "target_share", "air_yards_share",
    "depth_chart_position", "implied_team_total"
]].to_string())

      season  week team  targets  receptions  receiving_yards  receiving_tds  fantasy_points  target_share  air_yards_share  depth_chart_position  implied_team_total
3335    2020     1   GB       17          14              156              2            27.6      0.414634         0.416880                     1               23.00
3336    2020     2   GB        3           3               36              0             3.6      0.100000         0.075812                     1               22.00
3337    2020     6   GB       10           6               61              0             6.1      0.322581         0.258389                     1               26.00
3338    2020     7   GB       16          13              196              2            31.6      0.470588         0.575107                     1               26.50
3339    2020     8   GB       11           7               53              3            23.3      0.289474         0.380240                     1               20.75
3340

In [108]:
# Check a RB — Derrick Henry
sample_rb = df[df["player_display_name"] == "Derrick Henry"].head(10)
print(sample_rb[[
    "season", "week", "team", "carries", "rushing_yards", "rushing_tds",
    "fantasy_points", "depth_chart_position", "implied_team_total"
]].to_string())

      season  week team  carries  rushing_yards  rushing_tds  fantasy_points  depth_chart_position  implied_team_total
5842    2020     1  TEN       31            116            0            13.1                     1               19.00
5843    2020     2  TEN       25             84            0             8.4                     1               18.75
5844    2020     3  TEN       26            119            2            25.0                     1               23.25
5845    2020     5  TEN       19             57            2            18.3                     1               27.50
5846    2020     6  TEN       22            212            2            38.4                     1               24.00
5847    2020     7  TEN       20             75            1            13.2                     1               25.00
5848    2020     8  TEN       18            112            1            17.2                     1               21.00
5849    2020     9  TEN       21             68 